# Face Anti-Spoofing Training with MobileNetV2

This notebook trains a lightweight **MobileNetV2** model to distinguish between Real and Spoof faces.
It uses **LCC FASD** dataset downloaded automatically via `kagglehub`.

### Architecture
- **Backbone**: MobileNetV2 (Pre-trained on ImageNet)
- **Classifier**: Modified to output 2 classes (Real, Spoof)
- **Loss Function**: Weighted CrossEntropyLoss (to penalize mistakes on Real faces more)

### Metrics
- **Accuracy**: Can be misleading on imbalanced data.
- **F1-Score**: The harmonic mean of Precision and Recall. **Critical for this dataset** because of the imbalance (1 Real : 9 Spoof). We want to balance catching spoofs without rejecting too many real users.

In [ ]:
!pip install kagglehub scikit-learn
import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("aleksandrpikul222/lcc-fasd")

print("Path to dataset files:", path)

dataset_root = path
if os.path.exists(os.path.join(path, 'LCC_FASD')):
    dataset_root = os.path.join(path, 'LCC_FASD')
    
print(f"Dataset Root: {dataset_root}")

In [ ]:
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import numpy as np
from tqdm import tqdm
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix

print(f"PyTorch Version: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Define Dataset Class (LCC FASD Specific)


In [ ]:
class ASDataset(Dataset):
    def __init__(self, root_dir, client_file_name, imposter_file_name, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        
        client_path = os.path.join(root_dir, client_file_name)
        imposter_path = os.path.join(root_dir, imposter_file_name)
        
        with open(client_path, "r") as f:
            client_files = f.read().splitlines()
        with open(imposter_path, "r") as f:
            imposter_files = f.read().splitlines()
        
        self.imgs = []
        self.labels = []
        
        # Helper to fix path
        def clean_path(p):
            p = p.strip()
            if "LCC_FASD" in p:
                parts = p.split("LCC_FASD")
                rel = parts[-1].lstrip("/\\")
                return os.path.join(self.root_dir, rel)
            return os.path.join(self.root_dir, p)

        self.imgs.extend([clean_path(f) for f in client_files])
        self.labels.extend([1] * len(client_files)) # Real = 1
        
        self.imgs.extend([clean_path(f) for f in imposter_files])
        self.labels.extend([0] * len(imposter_files)) # Spoof = 0
        
        print(f"Loaded stats from {client_file_name} & {imposter_file_name}:")
        print(f"  Real (1): {len(client_files)}")
        print(f"  Spoof (0): {len(imposter_files)}")

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img_path = self.imgs[idx]
        try:
            img = Image.open(img_path).convert('RGB')
        except Exception as e:
            return self.__getitem__((idx + 1) % len(self))
            
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

## 2. Prepare Data Loaders


In [ ]:
# Config
BATCH_SIZE = 32
IMG_SIZE = 224

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Locate the Txt files
def find_file(root, name):
    for dirpath, dirnames, filenames in os.walk(root):
        for f in filenames:
            if name in f:
                return os.path.relpath(os.path.join(dirpath, f), root)
    return None

client_train = find_file(dataset_root, "CLIENT_TRAIN.txt") or "LCC_FASD_training/CLIENT_TRAIN.txt"
imposter_train = find_file(dataset_root, "IMPOSTER_TRAIN.txt") or "LCC_FASD_training/IMPOSTER_TRAIN.txt"

client_test = find_file(dataset_root, "CLIENT_TEST.txt") or "LCC_FASD_development/CLIENT_DEVELOPMENT.txt"
imposter_test = find_file(dataset_root, "IMPOSTER_TEST.txt") or "LCC_FASD_development/IMPOSTER_DEVELOPMENT.txt"

print(f"Using Train Files: {client_train}, {imposter_train}")
print(f"Using Val Files: {client_test}, {imposter_test}")

# Create Loaders
train_dataset = ASDataset(dataset_root, client_train, imposter_train, transform=train_transform)
val_dataset = ASDataset(dataset_root, client_test, imposter_test, transform=val_transform)

# Use WeightedRandomSampler for Training? 
# Alternatively, we used Weighted Loss. Let's stick to Weighted Loss for simplicity in code, 
# but note that Sampling is also a valid strategy.

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

## 3. Define MobileNetV2 Model


In [ ]:
def get_model(device):
    model = models.mobilenet_v2(pretrained=True)
    model.classifier[1] = nn.Linear(model.last_channel, 2)
    model = model.to(device)
    return model

model = get_model(device)

## 4. Handle Class Imbalance
**Class Weights**: Real (1) is minority (~1:9). We weight it 9.0 to 1.0.

In [ ]:
class_weights = torch.tensor([1.0, 9.0]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=0.0001)

## 5. Training Loop with F1-Score
We collect all predictions to calculate F1-Score at the end of each epoch.

In [ ]:
EPOCHS = 10
best_f1 = 0.0 # Track Best F1 instead of Accuracy
save_path = 'antispoof_model.pth'

for epoch in range(EPOCHS):
    print(f"Epoch {epoch+1}/{EPOCHS}")
    
    # --- Train ---
    model.train()
    running_loss = 0.0
    
    for images, labels in tqdm(train_loader, desc="Training"):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
            
    print(f"Train Loss: {running_loss/len(train_loader):.4f}")
    
    # --- Validation ---
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Validation"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    # Metrics Calculation
    accuracy = np.mean(np.array(all_preds) == np.array(all_labels))
    f1 = f1_score(all_labels, all_preds, average='binary') # Binary F1 for Class 1 (Real) usually, or weighted
    # Note: labels are 1 (Real), 0 (Spoof). Binary default pos_label=1.
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    
    print(f"Val Acc: {accuracy*100:.2f}%")
    print(f"Val F1-Score: {f1:.4f} (Precision: {precision:.4f}, Recall: {recall:.4f})")
    
    # Save based on F1-Score because of imbalance
    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), save_path)
        print(f"Saved Best Model (F1: {best_f1:.4f}) to {save_path}")

## 6. Export to ONNX


In [ ]:
dummy_input = torch.randn(1, 3, 224, 224).to(device)
torch.onnx.export(model, dummy_input, "antispoof_model.onnx", verbose=True)